# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiaburdibaloch-ui/flyrank-rafia/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import duckdb
import pandas as pd

In [2]:
import pandas as pd
import numpy as np

In [3]:
import duckdb

con = duckdb.connect()
print("DuckDB connected successfully")

DuckDB connected successfully


In [4]:
import os

path = "data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

File exists: False


In [5]:
import os

for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    if level <= 3:
        print(root, files[:10])

. []
./.config ['config_sentinel', 'gce', '.last_opt_in_prompt.yaml', '.last_survey_prompt.yaml', 'hidden_gcloud_config_universe_descriptor_data_cache_configs.db', 'active_config', 'default_configs.db', '.last_update_check.json']
./.config/logs []
./.config/logs/2026.09.16 ['13.26.04.283691.log', '13.26.17.091876.log', '13.26.29.139725.log', '13.26.30.459687.log', '13.26.15.129583.log', '13.25.37.877256.log']
./.config/configurations ['config_default']
./sample_data ['README.md', 'anscombe.json', 'mnist_test.csv', 'mnist_train_small.csv', 'california_housing_test.csv', 'california_housing_train.csv']


In [6]:
!git clone https://github.com/rafiaburdibaloch-ui/flyrank-rafia.git

Cloning into 'flyrank-rafia'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 143 (delta 52), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 9.94 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [7]:
import os

for root, dirs, files in os.walk("flyrank-rafia"):
    level = root.replace("flyrank-rafia", "").count(os.sep)

    if level <= 3:
        print(root)
        for file in files[:15]:
            print("   ", file)

flyrank-rafia
    01_first_look_and_discovery.ipynb
    LICENSE
    CLAUDE.md
    SETUP.md
    AGENTS.md
    README.md
    requirements.txt
    DATA_USE.md
    .gitignore
    GUIDE.md
flyrank-rafia/work
    README.md
    capstone_report_template.md
flyrank-rafia/work/notebooks
    w01_research_question.ipynb
    w03_data_contract.ipynb
    w03_feature_leakage_check.ipynb
    w04_signal_audit.ipynb
    w07_action_playbook.ipynb
    w06_validation_audit.ipynb
    w05_model.ipynb
    w02_ml_task_framing.ipynb
    w04_baseline_score.ipynb
    capstone.ipynb
flyrank-rafia/.github
flyrank-rafia/.github/workflows
    personalize.yml
    smoke-test.yml
    data-path-smoke.yml
flyrank-rafia/outputs
    refresh_queue_sample.csv
    model_report.md
flyrank-rafia/outputs/charts
    action_mix.svg
    top_feature_importance.svg
    trend_distribution.svg
    confidence_mix.svg
    top_reason_codes.svg
flyrank-rafia/.git
    HEAD
    config
    index
    packed-refs
    description
flyrank-rafia/.gi

In [8]:
from pathlib import Path

print(Path("flyrank-rafia/skills/flyrank/flyrank-data/SKILL.md").read_text())

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

In [9]:
id="v9k3zq"
import pandas as pd

file_path = "flyrank-rafia/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print(df.shape)

(30000, 44)


In [10]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
age_check = (
    df.groupby("age_tier")
    .agg(
        n=("content_id", "count"),
        avg_age_days=("content_age_days", "mean")
    )
    .reset_index()
)

age_check

,age_tier,n,avg_age_days
0,181-365,11368,281.140482
1,31-90,492,90.000000
2,365+,6360,462.690566
3,91-180,11780,127.507385


In [12]:
[c for c in df.columns if "refresh" in c.lower() or "flag" in c.lower()]

[]

In [13]:
from pathlib import Path

dictionary = Path("flyrank-rafia/docs/data-dictionary.md").read_text()

print(dictionary[:5000])

# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_id` | text | Pseudo

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.